# 03 — EDA no conjunto de treino (Fold 1)
Passos 3–4 do roteiro: visualizar, procurar transformações úteis (cestas, dummies), e analisar ausentes, duplicadas, outliers e inconsistências.

**Regras:**
- Usa **somente o treino do Fold 1** (nunca o teste; a validação seria tolerada, mas não é necessária).
- O treino salvo tem a classe minoritária **repetida** pelo oversampling, o que distorceria as estatísticas. Por isso a EDA usa apenas as **linhas originais** (`repetido == False`).
- Nada aqui é aplicado aos dados: só registra **decisões** para o pré-processamento.

In [ ]:
# Localiza a raiz do repositório (funciona no VS Code e no Colab) e a coloca no sys.path
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Raiz do projeto:", ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src import config as cfg
from src.split_data import load_partition, load_ids

sns.set_theme(style="whitegrid")
cfg.ensure_dirs()
FOLD = 1

X, y = load_partition(FOLD, "treino")
ids = load_ids()
ids = ids[(ids["fold"] == FOLD) & (ids["conjunto"] == "treino")].reset_index(drop=True)
assert len(ids) == len(X)                       # mesma ordem das linhas

orig = ~ids["repetido"].to_numpy()
tr = X[orig].copy().reset_index(drop=True)
tr[cfg.TARGET] = y[orig].map({1: cfg.POSITIVE_CLASS, 0: cfg.NEGATIVE_CLASS}).to_numpy()
print(f"Treino salvo: {len(X)} linhas | linhas originais usadas na EDA: {len(tr)} | % churn: {100*(tr[cfg.TARGET]=='Yes').mean():.2f}")

## 1. Visão geral e dados ausentes

In [ ]:
pd.DataFrame({"dtype": tr.dtypes.astype(str), "nunique": tr.nunique(), "ausentes": tr.isna().sum()})

In [ ]:
print("Ausentes em TotalCharges:", tr["TotalCharges"].isna().sum())
tr[tr["TotalCharges"].isna()][["tenure", "MonthlyCharges", "TotalCharges", "Contract", cfg.TARGET]]

## 2. Duplicadas e inconsistências

In [ ]:
print("Linhas duplicadas no treino (originais):", int(tr.duplicated().sum()))

# Inconsistências lógicas entre variáveis (esperado: 0 violações)
checks = {
    "PhoneService=No mas MultipleLines != 'No phone service'":
        ((tr["PhoneService"] == "No") & (tr["MultipleLines"] != "No phone service")).sum(),
    "InternetService=No mas algum serviço != 'No internet service'":
        ((tr["InternetService"] == "No") &
         (tr[["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]]
          .ne("No internet service").any(axis=1))).sum(),
    "tenure == 0 mas TotalCharges > 0": ((tr["tenure"] == 0) & (tr["TotalCharges"] > 0)).sum(),
    "tenure > 0 e TotalCharges == 0": ((tr["tenure"] > 0) & (tr["TotalCharges"] == 0)).sum(),
}
pd.Series(checks, name="violações")

## 3. Alvo

In [ ]:
tr[cfg.TARGET].value_counts()

## 4. Variáveis categóricas × churn
Taxa de churn por categoria (linha tracejada = taxa geral do treino).

In [ ]:
rate = (tr[cfg.TARGET] == "Yes").mean()
cols = cfg.CATEGORICAL_COLS
fig, axes = plt.subplots(4, 4, figsize=(18, 14))
for ax, col in zip(axes.ravel(), cols):
    t = tr.groupby(col)[cfg.TARGET].apply(lambda s: (s == "Yes").mean()).sort_values()
    sns.barplot(x=t.values, y=t.index.astype(str), ax=ax, color="tab:red")
    ax.axvline(rate, ls="--", c="k", lw=1); ax.set(title=col, xlabel="taxa de churn", ylabel="", xlim=(0, 0.8))
fig.tight_layout(); fig.savefig(cfg.REPORTS_FIGURES / "03_categoricas_vs_churn.png", dpi=130); plt.show()

## 5. Variáveis numéricas

In [ ]:
num = cfg.NUMERIC_COLS
tr[num].describe().T.to_csv(cfg.REPORTS_TABLES / "03_numericas_describe.csv")
tr[num].describe().T

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for j, col in enumerate(num):
    sns.histplot(data=tr, x=col, hue=cfg.TARGET, bins=40, kde=False, element="step", ax=axes[0, j],
                 palette={"No": "tab:blue", "Yes": "tab:red"})
    sns.boxplot(data=tr, x=cfg.TARGET, y=col, ax=axes[1, j], palette={"No": "tab:blue", "Yes": "tab:red"}, hue=cfg.TARGET, legend=False)
    axes[0, j].set_title(col)
fig.tight_layout(); fig.savefig(cfg.REPORTS_FIGURES / "03_numericas_dist.png", dpi=130); plt.show()

### Outliers (regra de Tukey: 1,5 × IQR)

In [ ]:
rows = []
for col in num:
    s = tr[col].dropna(); q1, q3 = s.quantile([.25, .75]); iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    rows.append({"coluna": col, "limite_inf": lo, "limite_sup": hi, "outliers": int(((s < lo) | (s > hi)).sum())})
pd.DataFrame(rows)

### Correlação (Spearman)

In [ ]:
corr_df = tr[num].copy()
corr_df["Churn"] = (tr[cfg.TARGET] == "Yes").astype(int)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(corr_df.corr(method="spearman"), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax)
fig.tight_layout(); fig.savefig(cfg.REPORTS_FIGURES / "03_correlacao_spearman.png", dpi=130); plt.show()

## 6. Candidatas a cestas (faixas) e variáveis dummy
Os slides sugerem transformar tempo de casa e valores de pagamento em faixas.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
tenure_bins = pd.cut(tr["tenure"], [-1, 6, 12, 24, 48, 72], labels=["0-6", "7-12", "13-24", "25-48", "49-72"])
t = tr.groupby(tenure_bins, observed=True)[cfg.TARGET].apply(lambda s: (s == "Yes").mean())
t.plot.bar(ax=axes[0], rot=0, color="tab:red", title="Churn por faixa de tenure (meses)")
mc = pd.qcut(tr["MonthlyCharges"], 5)
tr.groupby(mc, observed=True)[cfg.TARGET].apply(lambda s: (s == "Yes").mean()).plot.bar(
    ax=axes[1], rot=20, color="tab:red", title="Churn por quintil de MonthlyCharges")
tc = pd.qcut(tr["TotalCharges"].dropna(), 5)
tr.loc[tr["TotalCharges"].notna()].groupby(tc, observed=True)[cfg.TARGET].apply(lambda s: (s == "Yes").mean()).plot.bar(
    ax=axes[2], rot=20, color="tab:red", title="Churn por quintil de TotalCharges")
for ax in axes: ax.set_xlabel("")
fig.tight_layout(); fig.savefig(cfg.REPORTS_FIGURES / "03_faixas_vs_churn.png", dpi=130); plt.show()

## 7. Decisões para o pré-processamento (preencher/ajustar em grupo)
Registrar aqui o que foi decidido a partir dos gráficos acima. Sugestão de itens:
- `TotalCharges` ausente → imputar `0` quando `tenure == 0` (confirmado nas tabelas acima).
- Valores "No phone service" / "No internet service" → manter como categoria própria nos modelos de árvore; nos neurais, avaliar colapsar em "No".
- Numéricas (`tenure`, `MonthlyCharges`, `TotalCharges`) → Min-Max **ajustado só no treino**.
- Categóricas → one-hot (neurais); sem transformação numérica nominal para árvores.
- Variáveis agregadas/cestas candidatas: faixas de `tenure`, `MonthlyCharges`, total de serviços contratados.
- Não remover variáveis no primeiro modelo (exceto identificadores; não há).